In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN
import matplotlib.pyplot as plt
import yfinance as yf

In [2]:
# List of festivals with their dates
# Festivals that have already occurred in 2026 (as of April 19)
festivals = [
   {"name": "Makar Sankranti",           "date": "2026-01-14"},
   {"name": "Vasant Panchami",           "date": "2026-02-02"},
   {"name": "Maha Shivratri",            "date": "2026-02-15"},
   {"name": "Holika Dahan",              "date": "2026-03-03"},
   {"name": "Holi",                      "date": "2026-03-04"},
   {"name": "Eid al-Fitr",               "date": "2026-03-30"},
   {"name": "Good Friday",               "date": "2026-04-03"},
   {"name": "Easter",                    "date": "2026-04-05"},
   {"name": "Ram Navami",                "date": "2026-03-27"},
   {"name": "Baisakhi",                  "date": "2026-04-14"},
   {"name": "Vishu",                     "date": "2026-04-14"},
]

In [3]:
# Calculate date ranges for each festival
def get_date_range(festival_date):
   festival_date = datetime.strptime(festival_date, "%Y-%m-%d")
   start_date = festival_date - timedelta(days=10)
   end_date = festival_date + timedelta(days=10)
   return start_date, end_date
print(get_date_range('2026-01-14'))

(datetime.datetime(2026, 1, 4, 0, 0), datetime.datetime(2026, 1, 24, 0, 0))


In [4]:
# Prepare date ranges
date_ranges = [
   {
       "festival": festival["name"],
       "festival_date": festival["date"],
       "start_date": get_date_range(festival["date"])[0],
       "end_date": get_date_range(festival["date"])[1],
   }
   for festival in festivals
]

In [5]:
def fetch_and_train_model(date_range):
 print(
   f"Processing Festival: {date_range['festival']}, "
   f"Start Date: {date_range['start_date'].date()}, End Date: {date_range['end_date'].date()}"
 )
 try:
     # Fetch stock data
     data = yf.download(
         'MANAPPURAM.NS', start=date_range['start_date'].date(), end=date_range['end_date'].date()
     )
     data = data[['Close']]
     print(data)
     # Normalize the data
     scaler = MinMaxScaler(feature_range=(0, 1))
     scaled_data = scaler.fit_transform(data)  
     print("after normalization")
     print(scaled_data)
     # Prepare training data
     X = scaled_data[:-1]  # All except the last value as input
     y = scaled_data[1:]  # All except the first value as target
     X = X.reshape(X.shape[0], 1, 1)  # Reshape to (samples, timesteps, features)

     # Split into train and test sets
     train_size = int(len(X) * 0.8)
     X_train, X_test = X[:train_size], X[train_size:] # [0:11] -> Training, [11:end] -> Testing
     y_train, y_test = y[:train_size], y[train_size:]
     # Build the RNN model using SimpleRNN
     model = Sequential([
         SimpleRNN(50, activation='relu', return_sequences=True, input_shape=(X_train.shape[1], 1)),
         SimpleRNN(50, activation='relu'),  # Second SimpleRNN layer
         Dense(1)  # Output layer
     ])
     # Compile and train the model
     model.compile(loss='mean_squared_error', optimizer='adam')
     model.fit(X_train, y_train, epochs=50, batch_size=20, verbose=1)
     # Predict
     predicted_prices = model.predict(X_test)
     predicted_prices = scaler.inverse_transform(predicted_prices.reshape(-1, 1))
     actual_prices = scaler.inverse_transform(y_test.reshape(-1, 1))
     return data.index[-len(y_test):], actual_prices, predicted_prices
 except Exception as ex:
       print(f"Error processing {date_range['festival']}: {ex}")
       return [], [], []
data = fetch_and_train_model(date_ranges[0])
print(data)

Processing Festival: Makar Sankranti, Start Date: 2026-01-04, End Date: 2026-01-24


[*********************100%***********************]  1 of 1 completed


Price              Close
Ticker     MANAPPURAM.NS
Date                    
2026-01-05    306.093994
2026-01-06    307.240204
2026-01-07    318.852020
2026-01-08    308.486084
2026-01-09    284.913574
2026-01-12    293.236206
2026-01-13    307.040833
2026-01-14    307.987732
2026-01-15    307.987732
2026-01-16    312.971375
2026-01-19    313.419922
2026-01-20    300.861176
2026-01-21    297.571960
2026-01-22    299.216583
2026-01-23    293.784424
after normalization
[[0.62408337]
 [0.65785657]
 [1.        ]
 [0.69456656]
 [0.        ]
 [0.24522725]
 [0.65198207]
 [0.67988256]
 [0.67988256]
 [0.82672613]
 [0.83994263]
 [0.46989781]
 [0.37298073]
 [0.42143972]
 [0.26138055]]


c:\Users\jebarajj\OneDrive - Bristol Myers Squibb\Documents\Python\gen-ai\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - loss: 0.4770
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step - loss: 0.4465
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step - loss: 0.4174
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step - loss: 0.3903
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - loss: 0.3657
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - loss: 0.3424
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step - loss: 0.3200
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 111ms/step - loss: 0.2983
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - loss: 0.2782
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - loss: 0.2593
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step - loss: 0.2414
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - loss: 0.2249
Epoch 13/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step - loss: 0.2094
Epoch 14/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - loss: 0.1949
Epoch 15/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - loss: 0.1818
Epoch 16/50
1/1 ━━━━━━━